In [24]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta import *
from pyspark.sql.types import IntegerType, StructType, StructField
from pyspark.sql.types import TimestampType
from pyspark.sql.window import Window
import time

builder = SparkSession.builder.appName("OlistSilverTransformation") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

# Đường dẫn các tầng
bronze_root = "/home/jovyan/work/delta_lake/bronze"
silver_root = "/home/jovyan/work/delta_lake/silver"
checkpoint_silver = "/home/jovyan/work/delta_lake/_checkpoints/silver"

### BẢNG PRODUCTS
Từ bước khám phá dữ liệu, ta có các công việc cần làm tại tầng silver như sau: 
- 610 dòng null -> làm sạch null
- Dịch tên category thêm vào 2 category chưa có bản dịch tên tiếng anh

BATCH
- Giải thích mục đích batch: nói rõ rằng batch chỉ dùng để bootstrap bảng Silver ban đầu, tạo folder và dữ liệu khởi tạo để các bảng khác (như order_items) có thể join. Đây là bước khởi tạo cần thiết vì streaming không tạo folder nếu chưa có dữ liệu.

In [2]:
# Đọc Bronze
df_products_bronze_batch = spark.read.format("delta").load(f"{bronze_root}/products")
df_translations = spark.read.format("delta").load(f"{bronze_root}/translations")

# Khai báo schema giống với df_translations
schema = StructType([
    StructField("product_category_name", StringType(), True),
    StructField("product_category_name_english", StringType(), True),
    StructField("ingested_at", TimestampType(), True),
    StructField("source_file", StringType(), True)
])

# Tạo dữ liệu bổ sung với đủ 4 cột
extra_mapping = [
    ("pc_gamer", "pc_gamer", None, "manual_addition"),
    ("portateis_cozinha_e_preparadores_de_alimentos", "kitchen_portables", None, "manual_addition")
]

df_extra = spark.createDataFrame(extra_mapping, schema)

# Union sẽ hợp lệ vì cùng số cột và kiểu dữ liệu
df_translation_fixed = df_translations.union(df_extra)


# Loại bỏ null category
df_clean = df_products_bronze_batch.filter(F.col("product_category_name").isNotNull())

# Join với translation để lấy tên tiếng Anh
df_joined = df_clean.join(df_translation_fixed, on="product_category_name", how="left")

# Loại bỏ các cột kỹ thuật không cần thiết
df_joined = df_joined.drop("ingested_at", "source_file")

# Ép kiểu các cột số
df_casted = df_joined \
    .withColumn("product_photos_qty", F.col("product_photos_qty").cast(IntegerType())) \
    .withColumn("product_weight_g", F.col("product_weight_g").cast(IntegerType())) \
    .withColumn("product_length_cm", F.col("product_length_cm").cast(IntegerType())) \
    .withColumn("product_height_cm", F.col("product_height_cm").cast(IntegerType())) \
    .withColumn("product_width_cm", F.col("product_width_cm").cast(IntegerType()))

# Loại trùng lặp theo product_id
df_products_silver_batch = df_casted.dropDuplicates(["product_id"])

# Ghi xuống Silver
df_products_silver_batch.write.format("delta").mode("overwrite").save(f"{silver_root}/products") 

STREAMING
- Giải thích mục đích streaming: sau khi đã có folder, chuyển sang readStream + writeStream để pipeline tiếp tục ingest dữ liệu mới theo thời gian thực. Như vậy vẫn giữ được tính streaming của Lakehouse.

In [3]:
# Đọc Bronze
df_products_bronze = spark.readStream.format("delta").load(f"{bronze_root}/products")
df_translations = spark.read.format("delta").load(f"{bronze_root}/translations")

# Khai báo schema giống với df_translations
schema = StructType([
    StructField("product_category_name", StringType(), True),
    StructField("product_category_name_english", StringType(), True),
    StructField("ingested_at", TimestampType(), True),
    StructField("source_file", StringType(), True)
])

# Tạo dữ liệu bổ sung với đủ 4 cột
extra_mapping = [
    ("pc_gamer", "pc_gamer", None, "manual_addition"),
    ("portateis_cozinha_e_preparadores_de_alimentos", "kitchen_portables", None, "manual_addition")
]

df_extra = spark.createDataFrame(extra_mapping, schema)

# Union sẽ hợp lệ vì cùng số cột và kiểu dữ liệu
df_translation_fixed = df_translations.union(df_extra)


# Loại bỏ null category
df_clean = df_products_bronze.filter(F.col("product_category_name").isNotNull())

# Join với translation để lấy tên tiếng Anh
df_joined = df_clean.join(df_translation_fixed, on="product_category_name", how="left")

# Ép kiểu các cột số
df_casted = df_joined \
    .withColumn("product_photos_qty", F.col("product_photos_qty").cast(IntegerType())) \
    .withColumn("product_weight_g", F.col("product_weight_g").cast(IntegerType())) \
    .withColumn("product_length_cm", F.col("product_length_cm").cast(IntegerType())) \
    .withColumn("product_height_cm", F.col("product_height_cm").cast(IntegerType())) \
    .withColumn("product_width_cm", F.col("product_width_cm").cast(IntegerType()))

# Loại trùng lặp theo product_id
df_products_silver = df_casted.dropDuplicates(["product_id"])

# Ghi xuống Silver
query_products = df_products_silver.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{checkpoint_silver}/products") \
    .start(f"{silver_root}/products")

# Kiểm tra query đang chạy
spark.streams.active


### BẢNG CUSTOMERS

In [4]:
# Đọc Bronze
df_customers_bronze = spark.readStream.format("delta").load(f"{bronze_root}/customers")

# 1. Loại bỏ null (phòng ngừa, dù hiện tại không có)
df_clean = df_customers_bronze.filter(
    F.col("customer_id").isNotNull() & F.col("customer_unique_id").isNotNull()
)

# 2. Loại bỏ trùng lặp theo customer_id
df_dedup = df_clean.dropDuplicates(["customer_id"])

# 3. Ghi xuống Silver
query_customers = df_dedup.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{checkpoint_silver}/customers") \
    .start(f"{silver_root}/customers")


### BẢNG ORDERS 

In [5]:
# Đọc Bronze
df_orders_bronze = spark.readStream.format("delta").load(f"{bronze_root}/orders")

# 1. Lọc trạng thái đơn hàng
df_filtered = df_orders_bronze.filter(F.col("order_status") == "delivered")

# 2. Loại bỏ null order_id
df_clean = df_filtered.filter(F.col("order_id").isNotNull())

# 3. Ép kiểu thời gian
df_casted = df_clean \
    .withColumn("order_purchase_timestamp", F.col("order_purchase_timestamp").cast(TimestampType())) \
    .withColumn("order_approved_at", F.col("order_approved_at").cast(TimestampType())) \
    .withColumn("order_delivered_customer_date", F.col("order_delivered_customer_date").cast(TimestampType()))

# 4. Loại bỏ đơn hàng không hợp lệ
df_valid = df_casted.filter(
    (F.col("order_delivered_customer_date").isNotNull()) &
    (F.col("order_approved_at").isNotNull()) &
    (F.col("order_delivered_customer_date") >= F.col("order_purchase_timestamp"))
)

# 5. Loại trùng lặp theo order_id (phòng ngừa)
df_orders_silver = df_valid.dropDuplicates(["order_id"])

# 6. Ghi xuống Silver
query_orders = df_orders_silver.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{checkpoint_silver}/orders") \
    .start(f"{silver_root}/orders")


### BẢNG ORDER_ITEMS

In [6]:
# Đọc Bronze
df_order_items_bronze = spark.readStream.format("delta").load(f"{bronze_root}/order_items")
df_products_silver = spark.read.format("delta").load(f"{silver_root}/products")

# 1. Loại bỏ null và sản phẩm không tồn tại
df_clean = df_order_items_bronze.filter(
    F.col("order_id").isNotNull() & F.col("product_id").isNotNull()
)

# Inner join với products Silver để đảm bảo product_id hợp lệ
df_valid = df_clean.join(df_products_silver.select("product_id"), on="product_id", how="inner")

# ép kiểu timestamp cho ingested_at
df_valid = df_valid.withColumn("ingested_at", F.col("ingested_at").cast("timestamp"))

# Loại trùng lặp theo order_id và product_id
df_order_items_silver = df_valid.dropDuplicates(["order_id", "product_id"])

# Ghi xuống Silver dạng streaming
query_order_items = df_order_items_silver.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{checkpoint_silver}/order_items") \
    .start(f"{silver_root}/order_items")


# DEMO TIMETRAVEL

In [28]:
# ĐỌC SỐ DÒNG CUSTOMERS HIỆN TẠI
df_customers_silver = spark.read.format("delta").load(f"{silver_root}/customers")
print(f"Số dòng customers hiện tại: {df_customers_silver.count()}")

Số dòng customers hiện tại: 99441


In [29]:
# GIẢ LẬP LỖI BẰNG CÁCH GHI ĐÈ VÀO BẢNG CHỈ CÒN LẠI 3 DÒNG 
df_customers_silver.limit(3).write.format("delta").mode("overwrite").save(f"{silver_root}/customers")

In [31]:
# ĐỌC SỐ DÒNG CUSTOMERS HIỆN TẠI
df_customers_silver = spark.read.format("delta").load(f"{silver_root}/customers")
print(f"Số dòng customers hiện tại: {df_customers_silver.count()}")
df_customers_silver.show(5)

Số dòng customers hiện tại: 3
+--------------------+--------------------+------------------------+------------------+--------------+--------------------+--------------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|     customer_city|customer_state|         ingested_at|         source_file|
+--------------------+--------------------+------------------------+------------------+--------------+--------------------+--------------------+
|08ef50ea1923ce347...|337691bc9242e1d53...|                   11065|            santos|            SP|2026-05-09 04:55:...|file:///home/jovy...|
|84872d1894135d15c...|3e8f3e58aafa9bb38...|                    2376|         sao paulo|            SP|2026-05-09 04:55:...|file:///home/jovy...|
|7e6e52c237a37ca56...|3b885932732cd5252...|                   25545|sao joao de meriti|            RJ|2026-05-09 04:55:...|file:///home/jovy...|
+--------------------+--------------------+------------------------+------------------+-------------

In [33]:
#kiểm tra lại lịch sử để tìm version trước đó
history = spark.sql("DESCRIBE HISTORY delta.`/home/jovyan/work/delta_lake/silver/customers`")
history.show(5)

+-------+--------------------+------+--------+----------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|       operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+----------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      3|2026-05-09 09:27:...|  null|    null|           WRITE|{mode -> Overwrit...|null|    null|     null|          2|  Serializable|        false|{numFiles -> 1, n...|        null|Apache-Spark/3.3....|
|      2|2026-05-09 09:19:...|  null|    null|           WRITE|{mode -> Overwrit...|null|    null|     null|          1|  Serializable|        false|{numFiles -> 16, ...|        nu

In [34]:
# Quay lại version 0 
df_restored = spark.read.format("delta").option("versionAsOf", 0).load(f"{silver_root}/customers")
print(f"Số dòng customers sau khi khôi phục: {df_restored.count()}")

Số dòng customers sau khi khôi phục: 99441


In [36]:
df_restored.show(10)

+--------------------+--------------------+------------------------+------------------+--------------+--------------------+--------------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|     customer_city|customer_state|         ingested_at|         source_file|
+--------------------+--------------------+------------------------+------------------+--------------+--------------------+--------------------+
|08ef50ea1923ce347...|337691bc9242e1d53...|                   11065|            santos|            SP|2026-05-09 04:55:...|file:///home/jovy...|
|84872d1894135d15c...|3e8f3e58aafa9bb38...|                    2376|         sao paulo|            SP|2026-05-09 04:55:...|file:///home/jovy...|
|7e6e52c237a37ca56...|3b885932732cd5252...|                   25545|sao joao de meriti|            RJ|2026-05-09 04:55:...|file:///home/jovy...|
|928b905d5421373de...|87cb6f9be6c56d669...|                   29090|           vitoria|            ES|2026-05-09 04:55:...|file://